# Aprendizado de Máquina — Aula prática 06

## Pré-processamento e Pipelines

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Se você voltar aos notebooks das aulas anteriores, vai encontrar em quase todos a
mesma linha: um `StandardScaler` metido dentro de um `Pipeline`, com um comentário
ao lado prometendo que a Aula 06 explicaria o que ele fazia ali. Chegou a hora de
pagar essa dívida.

A regra cabe numa frase:

> **toda etapa que aprende alguma coisa dos dados faz parte do modelo — e portanto
> tem de acontecer dentro da dobra.**

Você provavelmente já ouviu isso antes, e talvez até acredite. O que quase ninguém
diz é **quanto** custa desobedecer, e a resposta surpreende: não é "sempre muito".
Um dos vazamentos contra os quais os livros mais advertem é numericamente
inofensivo; um outro, que quase ninguém menciona, fabrica $R^2 = 0{,}40$ a partir
de ruído puro. Vamos medir os dois — mais um terceiro que costuma passar em branco
— e só então montar a maquinaria que impede os três de acontecerem.

### Objetivos

Quando terminar, você deve conseguir:

- olhar para um método e dizer se a escala das covariáveis importa para ele, **e
  por quê**;
- pôr um número no custo de três vazamentos diferentes, e ordená-los por gravidade;
- reconhecer o vazamento por **agrupamento**, que nenhum `Pipeline` evita, e
  corrigi-lo com `GroupKFold`;
- montar um `Pipeline` e explicar por que ele torna o vazamento impossível por
  construção, e não por disciplina;
- usar `ColumnTransformer` quando cada grupo de colunas pede um tratamento próprio;
- tratar as escolhas do pré-processamento como hiperparâmetros, buscando-as junto
  com as do modelo;
- e diagnosticar um arquivo que chegou quebrado, que é como os arquivos costumam
  chegar.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

A lista é mais longa que a de costume, e as caras novas são todas peças de
pré-processamento: o `ColumnTransformer`, que aplica receitas diferentes a grupos
diferentes de colunas; o `SimpleImputer`, para os faltantes; o `OneHotEncoder`,
para as categóricas; e o `SelectKBest`, que na Seção 3 vai fazer o papel de cobaia
— é com ele que cometeremos, de propósito, o pior erro desta aula.

In [ ]:
import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (MinMaxScaler, OneHotEncoder, RobustScaler,
                                   StandardScaler)
from sklearn.tree import DecisionTreeRegressor

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. Quem é sensível à escala, e quem não é

"Padronize sempre" é o tipo de conselho que não ajuda, porque não diz para quem nem
por quê — e quem segue conselho sem entender acaba padronizando onde não precisava
e esquecendo onde precisava.

Dá para descobrir sozinho, e o teste é simples: pegamos um problema, trocamos a
unidade de **uma única coluna** — multiplicando-a por 1000, como quem passa de
metros para milímetros — e vemos quem muda de resposta. A informação nos dados
continua exatamente a mesma; só a régua mudou.

In [ ]:
rng = np.random.default_rng(0)
n, d = 400, 6


def alvo(M):
    return M[:, 0] + 0.5 * M[:, 1] - M[:, 2] + 0.3 * M[:, 3] * M[:, 4]


X = rng.normal(size=(n, d))
y = alvo(X) + rng.normal(0, 0.5, size=n)
X_te = rng.normal(size=(3000, d))
r_te = alvo(X_te)

# a mesma informacao, com a coluna 0 em outra unidade
escala = np.ones(d)
escala[0] = 1000.0
Xe, Xe_te = X * escala, X_te * escala

metodos = {
    "MQO": skl.LinearRegression(),
    "Ridge (alpha=100)": skl.Ridge(alpha=100.0),
    "arvore": DecisionTreeRegressor(max_depth=6, random_state=0),
    "KNN (k=10)": KNeighborsRegressor(n_neighbors=10),
}

linhas = []
for nome, m in metodos.items():
    a = np.mean((m.fit(X, y).predict(X_te) - r_te) ** 2)
    b = np.mean((m.fit(Xe, y).predict(Xe_te) - r_te) ** 2)
    linhas.append({"metodo": nome, "escala original": a, "coluna 0 x1000": b,
                   "mudou?": "sim" if abs(b - a) > 1e-6 * max(a, 1) else "nao"})

# o coeficiente da coluna 0 na Ridge, nas duas escalas (reescalado para comparar)
c1 = skl.Ridge(alpha=100.0).fit(X, y).coef_[0]
c2 = skl.Ridge(alpha=100.0).fit(Xe, y).coef_[0] * 1000
print(f"Ridge: coeficiente da coluna 0 = {c1:.4f} na escala original, "
      f"{c2:.4f} depois de multiplicar a coluna por 1000")
pd.DataFrame(linhas).set_index("metodo").round(4)

Quatro métodos, três comportamentos — e cada um tem uma explicação própria.

**O MQO ignora a mudança.** Multiplicar uma coluna por 1000 divide o coeficiente
dela por 1000, o produto $\beta_j x_j$ fica igual e a predição sai idêntica: os
mesmos $0{,}0914$ nas duas escalas. É o que se chama de *equivariância* por
reescala, e a consequência prática é que padronizar antes de um MQO puro não muda
predição nenhuma.

**A árvore também ignora.** Ela nunca soma colunas: só pergunta "$x_j \le t$?", e
multiplicar a coluna por 1000 multiplica o $t$ junto, sem trocar a ordem de
ninguém. $0{,}6032$ nas duas.

**A Ridge muda, e o coeficiente impresso conta a história.** A penalidade
$\lambda\|\beta\|^2$ soma coeficientes de colunas diferentes, e uma soma dessas só
faz sentido se as unidades forem comparáveis. Na escala original, o coeficiente da
coluna 0 é encolhido de $1{,}0$ para $0{,}79$. Com a coluna mil vezes maior, o
coeficiente correspondente é mil vezes menor, quase não pesa em $\|\beta\|^2$ — e
escapa quase intacto, em $0{,}99$. **A mesma coluna, a mesma informação, e uma
regularização completamente diferente**, só porque alguém anotou os dados em
milímetros.

**E o KNN muda muito**, de $0{,}35$ para $1{,}46$. A distância euclidiana soma as
coordenadas ao quadrado; uma coluna mil vezes maior domina a soma sozinha, e as
outras cinco deixam de existir.

Junte os quatro casos e a regra aparece: **padronize sempre que o método somar
coisas vindas de colunas diferentes** — numa penalidade (Ridge, Lasso), numa
distância (KNN, SVM, $k$-médias) ou numa projeção (PCA). Fora disso não faz mal,
mas também não faz nada.

Antes de rodar a próxima célula, vale arriscar um palpite — e repare que dá para
arriscá-lo sem ver o resultado, só olhando o mecanismo de cada método. O **Lasso**
penaliza $\lambda\sum_j|\beta_j|$, que também é uma soma de coeficientes de colunas
diferentes: mesma situação da Ridge, então deve mudar. A **floresta** é feita de
árvores, e árvores só comparam valores dentro de cada coluna: deve ser indiferente,
como a árvore isolada foi. Vamos conferir, olhando de novo para o coeficiente.

In [ ]:
for nome, m in [("Lasso (alpha=0,1)", skl.Lasso(alpha=0.1)),
                ("floresta (B=200)", RandomForestRegressor(n_estimators=200,
                                                           random_state=0, n_jobs=-1))]:
    risco_a = np.mean((m.fit(X, y).predict(X_te) - r_te) ** 2)
    coef_a = getattr(m, "coef_", None)
    risco_b = np.mean((m.fit(Xe, y).predict(Xe_te) - r_te) ** 2)
    coef_b = getattr(m, "coef_", None)

    print(f"{nome:20s} original {risco_a:.4f}   reescalado {risco_b:.4f}"
          f"   ({100 * (risco_b - risco_a) / risco_a:+.1f}%)")
    if coef_a is not None:
        print(f"{'':20s} coef da coluna 0: {coef_a[0]:.4f} -> {coef_b[0]:.6f}"
              f"   nao-nulos: {(coef_a != 0).sum()} -> {(coef_b != 0).sum()}")

Os dois palpites se confirmam, e o da floresta se confirma de um jeito forte: ela
dá $0{,}2421$ nas duas escalas — os mesmos quatro dígitos, não um valor parecido. A
indiferença que medimos na árvore isolada atravessa intacta a agregação de 200
delas.

O Lasso muda, e o coeficiente explica por quê: ele sai de $0{,}8846$ para
$0{,}000983$, exatamente mil vezes menor, como tem de ser para o produto
$\beta_0 x_0$ continuar o mesmo. Só que agora esse coeficiente entra em
$\sum_j|\beta_j|$ valendo quase nada — a coluna 0 ficou praticamente **isenta** da
penalidade, enquanto as outras cinco continuam pagando integralmente.

E aqui vem a parte que merece atenção: o risco **melhorou** 9,2% com a reescala, e
o número de coeficientes não nulos nem se mexeu (3 dos 6 nos dois casos). Isso não
é um argumento para não padronizar — é um lembrete de que a mudança foi arbitrária.
A coluna 0 é justamente a de maior efeito verdadeiro nesta simulação, então
isentá-la da penalidade calhou de ajudar. Aplique a mesma reescala a uma coluna
irrelevante e o efeito se inverte. Nos dois casos, quem decidiu não foi você: foi a
unidade de medida em que os dados chegaram.

---
## 3. Vazamento grave: escolher variáveis olhando tudo

Agora o experimento central da aula, e o cenário é deliberadamente cruel: $n = 60$
observações, $p = 3000$ covariáveis, e uma resposta $y$ sorteada **sem nenhuma
relação com $X$** — ruído puro. Não há o que aprender nesses dados. O $R^2$
verdadeiro de qualquer modelo é zero, e qualquer valor positivo que apareça é
ilusão, sem exceção.

A análise é a mesma nas duas versões: escolher as 20 covariáveis mais
correlacionadas com $y$, ajustar um MQO nelas e avaliar por validação cruzada. O
que muda é só *onde* a escolha acontece:

- **errado:** selecionar olhando o conjunto inteiro e *depois* rodar a validação
  cruzada;
- **certo:** pôr a seleção dentro do `Pipeline`, para que ela seja refeita em cada
  dobra, usando só o treino daquela dobra.

Uma linha de código de diferença. Veja quanto ela vale.

In [ ]:
# As duas medicoes saem do MESMO laco, com a mesma sequencia de numeros
# aleatorios que gerou a figura desta aula nas notas.
rng_v = np.random.default_rng(21)
n_v, d_v, k_v, n_rep = 60, 3000, 20, 40
dobras = skm.KFold(5, shuffle=True, random_state=0)
unidades = np.array([1, 50, 0.01, 5, 1, 200, 0.1, 2])
beta = np.r_[1.5, 0.02, 80, 0.3, -1.0, 0.005, 10, 0.4]

r2_errado, r2_certo = [], []
r2_escala_errada, r2_escala_certa = [], []
for _ in range(n_rep):
    Xv = rng_v.normal(size=(n_v, d_v))
    yv = rng_v.normal(size=n_v)                 # NENHUMA relacao com Xv

    # (1) ERRADO: seleciona olhando TODOS os dados, e so depois valida
    sel = SelectKBest(f_regression, k=k_v).fit(Xv, yv)
    r2_errado.append(skm.cross_val_score(skl.LinearRegression(), sel.transform(Xv),
                                         yv, cv=dobras, scoring="r2").mean())

    # (2) CERTO: a selecao e' uma etapa do pipeline, refeita dentro de cada dobra
    pipe_sel = Pipeline([("sel", SelectKBest(f_regression, k=k_v)),
                         ("mqo", skl.LinearRegression())])
    r2_certo.append(skm.cross_val_score(pipe_sel, Xv, yv, cv=dobras,
                                        scoring="r2").mean())

    # (3) e (4): o mesmo par, com padronizacao no lugar da selecao, num
    #            problema pequeno, com sinal de verdade e escalas dispares
    Xp = rng_v.normal(size=(60, 8)) * unidades
    yp = Xp @ beta + rng_v.normal(0, 1, 60)

    esc = StandardScaler().fit(Xp)              # ERRADO: aprende com tudo
    r2_escala_errada.append(skm.cross_val_score(
        skl.Ridge(alpha=1.0), esc.transform(Xp), yp, cv=dobras, scoring="r2").mean())
    r2_escala_certa.append(skm.cross_val_score(
        Pipeline([("sc", StandardScaler()), ("ridge", skl.Ridge(alpha=1.0))]),
        Xp, yp, cv=dobras, scoring="r2").mean())

print(f"R^2 verdadeiro                     :  0.000")
print(f"selecao FORA da dobra (errado)     : {np.mean(r2_errado):+.3f}")
print(f"selecao DENTRO do pipeline (certo) : {np.mean(r2_certo):+.3f}")

> **Pare um instante neste número.** Um $R^2$ de $+0{,}40$ saiu **do nada**. Não
> existe sinal nenhum nesses dados — nós mesmos os geramos —, e ainda assim o
> procedimento errado reporta um modelo que explica 40% da variância. É um número
> que ninguém questionaria num relatório, num artigo ou numa reunião.
>
> O mecanismo é este: a seleção olhou 3000 colunas de ruído e ficou com as 20 que,
> *por acaso*, mais se pareciam com $y$ **naquela amostra**. Quando as dobras da
> validação cruzada chegaram, essas 20 já estavam escolhidas — e escolhidas usando,
> inclusive, as observações que deveriam fazer o papel de "novas". O modelo não
> aprendeu nada sobre o mundo; aprendeu sobre o próprio conjunto de validação.
>
> Feito certo, a validação cruzada devolve $R^2$ **negativo**, $-0{,}69$, e
> denuncia o que está havendo: prever pela média seria melhor do que usar este
> modelo.

---
## 4. Vazamento leve: padronizar antes de separar

Este é o vazamento que os livros mais advertem — e o laço da seção anterior já o
mediu de passagem, no mesmo `for`. O cenário foi montado para favorecer o alarme ao
máximo: um problema com sinal de verdade e oito colunas cujas escalas diferem por
quatro ordens de grandeza, que é exatamente a situação em que a padronização mais
importa.

In [ ]:
print(f"escala FORA da dobra (errado) : R^2 = {np.mean(r2_escala_errada):.4f}")
print(f"escala DENTRO do pipeline     : R^2 = {np.mean(r2_escala_certa):.4f}")
print(f"diferenca                     : {abs(np.mean(r2_escala_errada) - np.mean(r2_escala_certa)):.2e}")

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(7.6, 3.0))
m1 = [np.mean(r2_errado), np.mean(r2_certo)]
ax1.axhline(0, color="black", lw=0.8)
ax1.bar([0, 1], m1, width=0.55, color=["crimson", "green"])
ax1.set_xticks([0, 1]); ax1.set_xticklabels(["selecao\nfora da dobra",
                                             "selecao\ndentro do pipeline"], fontsize=8)
ax1.set_ylabel("R^2 estimado por CV")
ax1.set_title(f"y e' ruido puro (n={n_v}, d={d_v})", fontsize=9)
for xx, vv in zip([0, 1], m1):
    ax1.annotate(f"{vv:+.2f}", xy=(xx, vv), xytext=(0, 6 if vv > 0 else -14),
                 textcoords="offset points", ha="center", fontsize=9)

m2 = [np.mean(r2_escala_errada), np.mean(r2_escala_certa)]
ax2.bar([0, 1], m2, width=0.55, color=["crimson", "green"])
ax2.set_xticks([0, 1]); ax2.set_xticklabels(["escala\nfora da dobra",
                                             "escala\ndentro do pipeline"], fontsize=8)
ax2.set_ylabel("R^2 estimado por CV")
ax2.set_title("padronizacao, com sinal de verdade", fontsize=9)
ax2.set_ylim(min(m2) - 0.02, max(m2) + 0.02)
for xx, vv in zip([0, 1], m2):
    ax2.annotate(f"{vv:.4f}", xy=(xx, vv), xytext=(0, 6), textcoords="offset points",
                 ha="center", fontsize=9)

Iguais até a quarta casa decimal.

Não é que o aviso clássico esteja errado — é que ele mira no lugar errado. A
padronização vaza duas estatísticas por coluna, cada uma calculada sobre dezenas de
observações, e trocar as do treino pelas do conjunto todo mexe nelas por algo
desprezível. Já a seleção de variáveis vaza **a decisão de quais colunas olhar**, e
essa decisão pode ser inteiramente ditada pelo acaso da amostra em que foi tomada.

A moral não é "relaxe com a padronização". É: **corrija assim mesmo, porque custa
zero** — é só pôr no `Pipeline` —, mas guarde sua vigilância para o erro que cobra
caro.

---
## 5. O vazamento que ninguém vê: observações agrupadas

Existe um terceiro tipo, e é o mais traiçoeiro dos três, porque não mora em etapa
nenhuma de pré-processamento — e portanto escapa de qualquer `Pipeline` que você
monte. É simplesmente **a mesma unidade aparecer no treino e no teste**.

A situação é corriqueira: cinco medições do mesmo paciente, doze compras do mesmo
cliente, trinta fotos do mesmo indivíduo. Um `train_test_split` aleatório espalha
as medições de um paciente pelos dois lados da divisão, e a partir daí o modelo não
precisa aprender nada sobre pacientes em geral — basta reconhecer *aquele*
paciente, que ele já viu.

Vamos construir o caso com todas as cartas na mesa: 100 unidades, 8 medições cada,
e uma resposta que depende fortemente de um efeito próprio da unidade.

In [ ]:
rng_g = np.random.default_rng(7)
n_unidades, por_unidade = 100, 8
grupo = np.repeat(np.arange(n_unidades), por_unidade)

efeito_unidade = rng_g.normal(0, 2.0, size=n_unidades)      # o que nao generaliza
Xg = rng_g.normal(size=(len(grupo), 5))
yg = 0.8 * Xg[:, 0] + efeito_unidade[grupo] + rng_g.normal(0, 0.3, size=len(grupo))

modelo_g = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)

ingenua = skm.cross_val_score(modelo_g, np.c_[Xg, grupo], yg,
                              cv=skm.KFold(5, shuffle=True, random_state=0),
                              scoring="r2").mean()
honesta = skm.cross_val_score(modelo_g, np.c_[Xg, grupo], yg,
                              cv=skm.GroupKFold(5), groups=grupo,
                              scoring="r2").mean()
print(f"CV com KFold      (a unidade aparece nos dois lados): R^2 = {ingenua:+.3f}")
print(f"CV com GroupKFold (cada unidade so de um lado)      : R^2 = {honesta:+.3f}")

A primeira estimativa é pura fantasia. O modelo recebeu a coluna `grupo`, decorou o
efeito de cada unidade e reencontrou as mesmas unidades na validação, onde bastou
consultar a memória. A segunda responde à pergunta que a análise realmente fazia —
*o que este modelo faz diante de uma unidade que ele nunca viu?* — e a resposta é
bem menos animadora.

Repare que **nenhum `Pipeline` teria salvado você aqui**. O vazamento não está em
transformação alguma; está em como as dobras foram formadas. A ferramenta certa é o
`GroupKFold` — ou `StratifiedGroupKFold`, ou `TimeSeriesSplit` para dados temporais
—, e a pergunta que a dispara é sempre a mesma: *duas linhas do meu banco podem se
referir à mesma coisa do mundo?*

### A hierarquia que vale memorizar

Com as três medições na mão, dá para ordenar por gravidade — e a ordem não é a que
a maioria dos avisos sugere:

1. **Grave** — escolher variáveis, hiperparâmetros ou o modelo olhando dados que
   depois vão avaliá-lo. A inflação não tem teto: a Seção 3 fabricou
   $R^2 = 0{,}40$ a partir de ruído puro.
2. **Grave** — a mesma unidade nos dois lados da divisão, ou informação do futuro
   em dados temporais. É a Seção 5, e nenhum `Pipeline` protege.
3. **Leve** — padronizar, imputar pela média ou codificar categóricas usando o
   conjunto todo. Numericamente inofensivo com $n$ razoável, como a Seção 4 mediu —
   mas corrija assim mesmo, porque custa zero.

O que separa os graves do leve não é quanto a etapa aprende dos dados. É se ela
**olha o $Y$**. Sem esse canal, nenhuma transformação consegue fabricar acerto a
partir de ruído.

Uma objeção razoável neste ponto: e se a gente simplesmente não desse ao modelo a
coluna `grupo`? Sem a chave, ele não teria como reconhecer ninguém. Vale testar — o
efeito da unidade continua nos dados, o que sai é só o identificador.

In [ ]:
for rotulo, M in [("com a coluna grupo", np.c_[Xg, grupo]), ("sem a coluna grupo", Xg)]:
    ing = skm.cross_val_score(modelo_g, M, yg,
                              cv=skm.KFold(5, shuffle=True, random_state=0),
                              scoring="r2").mean()
    hon = skm.cross_val_score(modelo_g, M, yg, cv=skm.GroupKFold(5), groups=grupo,
                              scoring="r2").mean()
    print(f"{rotulo:20s} KFold {ing:+.4f}   GroupKFold {hon:+.4f}"
          f"   distancia {ing - hon:.4f}")

**A distância praticamente some: de $0{,}404$ para $0{,}023$.**

Sem a coluna `grupo`, a floresta não tem por onde decorar quem é quem. O efeito de
unidade nesta simulação é um sorteio $N(0;\,2^2)$ **independente** de $X$ — nada
nas cinco colunas o denuncia —, então não sobra o que memorizar, e as duas
validações cruzadas passam a concordar. Concordam, aliás, em dizer que o modelo não
presta: $R^2$ de $-0{,}002$ e $-0{,}025$, ambos em torno de zero, porque o efeito de
unidade era a maior parte da variância de $y$ e ficou inexplicável.

Mas cuidado com a moral fácil — "então basta remover a coluna de identificação".
Ela vale **aqui**, e vale porque nós construímos $X$ independente da unidade. Em
dados reais, o $X$ costuma carregar a assinatura da unidade sem pedir licença:
várias medições do mesmo paciente trazem a mesma idade, o mesmo sexo e o mesmo
histórico; várias fotos do mesmo animal trazem o mesmo fundo. Nesses casos, apagar
o identificador **não** apaga o vazamento, e a única defesa continua sendo dividir
por grupo.

A regra que sobrevive aos dois casos é a da célula anterior, e note que ela não fala
de colunas: *duas linhas do meu banco podem se referir à mesma coisa do mundo?* Se
sim, `GroupKFold`.

---
## 6. O `Pipeline`, por dentro

Já usamos o `Pipeline` três vezes nesta aula como quem usa uma caixa-preta
confiável. Vale abrir a caixa, porque o que ele garante é bem específico: uma
disciplina sobre quem chama `fit` em quê.

- `pipe.fit(X_tr, y_tr)` — cada transformador faz `fit_transform` **no treino**, em
  sequência, e o estimador final é ajustado no resultado;
- `pipe.predict(X_te)` — cada transformador apenas `transform`, com os parâmetros
  que aprendeu no treino, e o estimador prediz.

O teste nunca entra em nenhum `fit`. Isso não é promessa de boa conduta, é
consequência da estrutura — e dá para verificar, abrindo o scaler e perguntando a
ele o que aprendeu.

In [ ]:
X_tr, X_va, y_tr, y_va = skm.train_test_split(Xe, y, test_size=0.3, random_state=0)

pipe = Pipeline([("escala", StandardScaler()), ("ridge", skl.Ridge(alpha=1.0))])
pipe.fit(X_tr, y_tr)

media_aprendida = pipe.named_steps["escala"].mean_
print("media que o scaler aprendeu (3 primeiras colunas):",
      np.round(media_aprendida[:3], 4))
print("media do TREINO                                  :",
      np.round(X_tr.mean(axis=0)[:3], 4))
print("media do conjunto TODO                           :",
      np.round(Xe.mean(axis=0)[:3], 4))
print(f"\nbate com o treino? {np.allclose(media_aprendida, X_tr.mean(axis=0))}")

A média guardada pelo scaler é a do treino, não a do conjunto todo — e as duas são
bem diferentes na primeira coluna, $-33{,}94$ contra $-89{,}02$, porque é a coluna
que multiplicamos por 1000 lá na Seção 2.

E o `Pipeline` inteiro é um estimador como outro qualquer: entra em
`cross_val_score`, em `GridSearchCV`, dentro de um `BaggingRegressor`. É essa
composicionalidade que torna a coisa prática. A alternativa seria refazer o
pré-processamento à mão dentro de cada dobra — e é exatamente aí que nascem os erros
da Seção 3.

---
## 7. `ColumnTransformer`: colunas diferentes, tratamentos diferentes

Todos os conjuntos de dados deste repositório são numéricos e completos, o que é uma
sorte que você não vai ter sempre — na verdade, é uma sorte que você quase nunca vai
ter. Vamos então fabricar um banco sujo, com colunas categóricas, faltantes de
verdade e escalas díspares, e montar nele o fluxo completo.

In [ ]:
rng_s = np.random.default_rng(11)
n_s = 900
cidades = np.array(["Niteroi", "Rio", "Sao Goncalo", "Marica"])

df = pd.DataFrame({
    "idade": rng_s.integers(18, 80, n_s).astype(float),
    "renda": np.round(rng_s.lognormal(8.2, 0.6, n_s), 2),
    "cidade": rng_s.choice(cidades, n_s, p=[0.4, 0.3, 0.2, 0.1]),
    "plano": rng_s.choice(["basico", "pleno", "premium"], n_s),
})
alvo_s = (0.05 * df["idade"] + 0.0008 * df["renda"]
          + df["plano"].map({"basico": 0.0, "pleno": 1.5, "premium": 3.0})
          + rng_s.normal(0, 0.8, n_s))

# faltantes de verdade: 8% da renda e 5% da idade
df.loc[rng_s.random(n_s) < 0.08, "renda"] = np.nan
df.loc[rng_s.random(n_s) < 0.05, "idade"] = np.nan

print(df.dtypes.to_string())
print(f"\nfaltantes por coluna:\n{df.isna().sum().to_string()}")
df.head()

Repare no impasse que esse banco cria: o `SimpleImputer` só serve para as
numéricas, o `OneHotEncoder` só para as categóricas, e o `StandardScaler`
simplesmente quebra diante de uma coluna de texto. Não existe uma receita única que
sirva para as quatro colunas. O `ColumnTransformer` resolve isso do jeito óbvio —
aplicando cada receita ao seu próprio grupo de colunas.

In [ ]:
numericas = ["idade", "renda"]
categoricas = ["cidade", "plano"]

prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
])

modelo = Pipeline([("prep", prep), ("ridge", skl.Ridge(alpha=1.0))])

X_tr, X_te2, y_tr2, y_te2 = skm.train_test_split(df, alvo_s, test_size=0.3,
                                                 random_state=0)
modelo.fit(X_tr, y_tr2)
print("colunas depois do pre-processamento:",
      list(modelo.named_steps["prep"].get_feature_names_out()))
print(f"\nEQM no teste: {mean_squared_error(y_te2, modelo.predict(X_te2)):.4f}")
print(f"variancia de y: {y_te2.var():.4f}")

Duas colunas numéricas continuaram duas; as duas categóricas viraram sete
indicadoras, uma por nível.

O `handle_unknown="ignore"` não é preciosismo. Uma categoria pode existir no teste e
não no treino — basta a divisão dar azar com "Maricá", que aparece em 10% das linhas
—, e sem esse argumento o `predict` levanta exceção. Não aqui, onde tudo é pequeno e
você estaria olhando: em produção, meses depois, quando ninguém mais lembrar por
quê.

Note também a estrutura: o `ColumnTransformer` é ele próprio uma etapa de um
`Pipeline` maior, e dentro dele há outro `Pipeline` aninhado, para as numéricas. A
recursão é de propósito — é o que permite que qualquer análise, por mais complicada,
caiba num único objeto com um `fit` e um `predict`.

---
## 8. Buscar hiperparâmetros do pré-processamento

Se o pré-processamento é parte do modelo, então as decisões dele — mediana ou média
na imputação? `StandardScaler` ou `RobustScaler`? — são hiperparâmetros como
qualquer outro. E hiperparâmetro se escolhe por validação cruzada, não por hábito
nem pelo que estava no notebook de onde você copiou.

A sintaxe usa `__` para descer na hierarquia: `prep__num__imp__strategy` quer dizer
*"o parâmetro `strategy` da etapa `imp`, que está dentro da etapa `num`, que está
dentro da etapa `prep`"*.

In [ ]:
grade = {
    "prep__num__imp__strategy": ["mean", "median", "most_frequent"],
    "prep__num__sc": [StandardScaler(), MinMaxScaler(), RobustScaler()],
    "ridge__alpha": np.logspace(-2, 3, 12),
}

busca = skm.GridSearchCV(modelo, grade, cv=skm.KFold(5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error", n_jobs=-1)
busca.fit(X_tr, y_tr2)

print("melhores escolhas:")
for chave, valor in busca.best_params_.items():
    print(f"   {chave:28s} = {valor}")
print(f"\nEQM de CV no vencedor: {-busca.best_score_:.4f}")
print(f"EQM no teste         : {mean_squared_error(y_te2, busca.predict(X_te2)):.4f}")
print(f"combinacoes avaliadas: {len(busca.cv_results_['mean_test_score'])}")

In [ ]:
res = pd.DataFrame(busca.cv_results_)
res["escala"] = res["param_prep__num__sc"].astype(str).str.replace("()", "", regex=False)
resumo = (res.groupby(["escala", "param_prep__num__imp__strategy"])["mean_test_score"]
          .max().unstack() * -1)
resumo.round(4)

A tabela separa duas perguntas que costumam ser feitas juntas, e mostra que só uma
delas importa aqui.

**Qual `Scaler`** não faz diferença: as três linhas coincidem até a terceira casa
decimal. Se você já perdeu tempo decidindo entre `StandardScaler` e `RobustScaler`,
este é o tamanho do prêmio.

**Qual imputação**, faz muita. Preencher os faltantes de uma variável contínua com a
*moda* é uma ideia ruim, e custa 40% de erro a mais — $1{,}2254$ contra $0{,}8736$.
Ninguém precisou avisar a validação cruzada disso: ela descobriu sozinha, que é
justamente o argumento para pôr o pré-processamento na grade em vez de decidi-lo de
antemão.

E tudo isso aconteceu **dentro** da validação cruzada: cada uma das 108 combinações
foi avaliada refazendo imputação e escala em cada dobra. Fazer o mesmo à mão, sem
`Pipeline`, é o caminho mais curto para o erro da Seção 3.

Uma etapa de seleção de variáveis também é um hiperparâmetro, e entra na grade como
os outros. O que ela faz com o **tamanho** da busca, porém, merece um olhar — e é o
assunto da próxima célula.

In [ ]:
import time

grade_maior = dict(grade, sel__k=[2, 3, 4, 5, 6, 7])
modelo_sel = Pipeline([("prep", prep), ("sel", SelectKBest(f_regression)),
                       ("ridge", skl.Ridge())])

for rotulo, mod_b, gr in [("sem SelectKBest", modelo, grade),
                          ("com SelectKBest", modelo_sel, grade_maior)]:
    t0 = time.perf_counter()
    b = skm.GridSearchCV(mod_b, gr, cv=skm.KFold(5, shuffle=True, random_state=0),
                         scoring="neg_mean_squared_error", n_jobs=-1).fit(X_tr, y_tr2)
    print(f"{rotulo:16s} {len(b.cv_results_['mean_test_score']):4d} combinacoes"
          f"   {time.perf_counter() - t0:5.1f}s   EQM de CV {-b.best_score_:.4f}"
          f"   k = {b.best_params_.get('sel__k', '-')}")

t0 = time.perf_counter()
aleatoria = skm.RandomizedSearchCV(
    modelo_sel, grade_maior, n_iter=30, random_state=0,
    cv=skm.KFold(5, shuffle=True, random_state=0),
    scoring="neg_mean_squared_error", n_jobs=-1).fit(X_tr, y_tr2)
print(f"{'aleatoria (30)':16s} {30:4d} combinacoes"
      f"   {time.perf_counter() - t0:5.1f}s   EQM de CV {-aleatoria.best_score_:.4f}")

Seis valores de `k` multiplicam a grade por seis: de **108** para **648**
combinações, cada uma com 5 dobras, o que dá 3240 ajustes. O relógio cresce mais que
proporcionalmente — de menos de um segundo para cerca de dez —, porque a própria
etapa de seleção tem custo.

E o que se ganha com isso: $0{,}8592$ contra $0{,}8736$ de erro de validação
cruzada, **1,6%**. O `k` escolhido é 4, das 9 colunas que o pré-processamento
produz — quatro colunas fazendo o trabalho de nove.

A última linha é o ponto. O `RandomizedSearchCV` sorteia 30 das 648 combinações e
chega a $0{,}8620$ em dois décimos de segundo: fica $0{,}3\%$ atrás do ótimo da
busca exaustiva, ainda à frente da grade **sem** `SelectKBest`, e é **dezenas de
vezes** mais rápido. A diferença que ele deixa na mesa não sobreviveria a uma troca
de semente.

Os tempos variam de máquina para máquina e de rodada para rodada, então não se
preocupe se os seus não baterem com estes — o que não varia é a ordem de grandeza da
diferença.

A intuição por trás disso vem do [ISLP] e vale guardar: numa grade grande, quase
sempre **poucos hiperparâmetros importam de fato**, e a busca exaustiva gasta a
maior parte do tempo variando com precisão justamente os que não fazem diferença.
Sortear cobre a faixa dos que importam com muito menos ajustes. Aqui, com uma
diferença de segundos, tanto faz; com um modelo que leva minutos por ajuste, é a
diferença entre fazer e não fazer.

---
## 9. Um arquivo real, quebrado

Fechamos com um problema que nenhum livro cobre e que todo mundo enfrenta mais cedo
ou mais tarde: o arquivo chega errado. O `bank_train_redux.csv` deste repositório —
o mesmo que a Aula 08 vai usar — tem um defeito de exportação, e vale diagnosticá-lo
com calma antes de sair consertando.

Vamos ler só as primeiras linhas, porque o arquivo inteiro tem cerca de 100 MB.

In [ ]:
import os

_nome = "bank_train_redux.csv"

# procura em dois lugares, sem baixar nada da internet: a pasta deste
# notebook primeiro ou então ../../recursos/dados/
_lugares = [_nome, os.path.join("..", "..", "recursos", "dados", _nome)]
_caminho = next((c for c in _lugares if os.path.exists(c)), None)

if _caminho is None:
    raise FileNotFoundError(
        f"nao encontrei '{_nome}'. Procurei nesta pasta e em "
        "../../recursos/dados/. Ponha o .csv ao lado deste notebook, "
        "ou mude o caminho se for necessário."
    )

banco = pd.read_csv(_caminho, nrows=20_000)
print("dimensoes:", banco.shape)
print("\nultimas 3 colunas:", list(banco.columns)[-3:])
print("\ntipos das ultimas 3 colunas:")
print(banco.dtypes.tail(3).to_string())

Dois sintomas, e eles apontam para a mesma causa. O nome da última coluna é
`var_199;;;;;;;` — sobraram separadores no cabeçalho, como se a linha tivesse sido
escrita prevendo colunas que nunca vieram. E o `dtype` dela é `object`, isto é,
texto, enquanto as outras 199 são numéricas.

Quando uma coluna que deveria ser número chega como texto, quase sempre há sujeira
dentro dos valores. Vale olhar.

In [ ]:
coluna = banco.columns[-1]
print("tres valores dessa coluna, como vieram:")
print(banco[coluna].head(3).to_string())
print(f"\ntentando converter direto: ", end="")
try:
    pd.to_numeric(banco[coluna])
    print("funcionou")
except ValueError as erro:
    print(f"ValueError -> {str(erro)[:70]}")

In [ ]:
limpo = banco.copy()
limpo = limpo.replace(to_replace=";", value="", regex=True)
limpo = limpo.rename(columns={coluna: "var_199"})
limpo["var_199"] = pd.to_numeric(limpo["var_199"])

print(f"colunas nao numericas antes : {(banco.dtypes == object).sum()}")
print(f"colunas nao numericas depois: {(limpo.dtypes == object).sum()}"
      f"   (a ID_code, que e' texto de verdade)")
print(f"\nvar_199 agora: media {limpo['var_199'].mean():.3f}, "
      f"desvio {limpo['var_199'].std():.3f}")
print(f"prevalencia da classe positiva: {limpo['target'].mean():.4f}")

Três linhas resolveram. Mas pense no que aconteceria sem o diagnóstico: a coluna
`var_199` seria descartada em silêncio pelo `ColumnTransformer` — ou, pior, viraria
um punhado de *dummies* de texto no `OneHotEncoder` —, e ninguém notaria.

Repare também onde essa limpeza **não** entra: ela não depende dos dados de treino,
é uma correção de formato do arquivo. A mesma substituição valeria para qualquer
amostra, de qualquer tamanho. Por isso faz sentido fazê-la uma vez, antes de tudo. A
distinção é essa, e é a da aula inteira: o que **aprende parâmetros dos dados** vai
para o `Pipeline`; o que é conserto determinístico de formato pode acontecer antes.

Com o arquivo em ordem, montamos o `Pipeline` completo: fora a `ID_code`, que é
texto de identificação e não covariável; padronização das 200 colunas `var_*`; e uma
regressão logística no fim. A avaliação vai por validação cruzada de 5 dobras
**estratificadas** — e a estratificação importa aqui, porque a classe positiva é
rara.

In [ ]:
Xb = limpo.drop(columns=["ID_code", "target"])
yb = limpo["target"].values

tubo_banco = Pipeline([("escala", StandardScaler()),
                       ("logistica", skl.LogisticRegression(max_iter=2000))])
dobras_b = skm.StratifiedKFold(5, shuffle=True, random_state=0)

acc = skm.cross_val_score(tubo_banco, Xb, yb, cv=dobras_b, scoring="accuracy")
auc = skm.cross_val_score(tubo_banco, Xb, yb, cv=dobras_b, scoring="roc_auc")

print(f"n = {len(yb)}, covariaveis = {Xb.shape[1]}, prevalencia = {yb.mean():.4f}\n")
print(f"acuracia por dobra: {np.round(acc, 4)}")
print(f"acuracia media    : {acc.mean():.4f}")
print(f"AUC media         : {auc.mean():.4f}")
print(f"\nacuracia de quem responde SEMPRE 'nao': {1 - yb.mean():.4f}")

**Acurácia de $0{,}9117$.** Parece excelente. Guarde o número por um instante.

Agora olhe a última linha: um modelo que responde "não" para todo mundo, sem
consultar covariável nenhuma, acerta $0{,}9016$. O nosso `Pipeline`, com 200 colunas
e uma regressão logística, comprou **um ponto percentual** sobre isso.

A acurácia não está mentindo; ela está respondendo a uma pergunta que não é a
interessante. Quando 90% das respostas são "não", acertar 90% é o piso, não o teto, e
a métrica mal distingue um modelo útil de um modelo mudo.

A AUC de $0{,}8479$ já conta outra história. Ela mede a capacidade de **ordenar** — a
probabilidade de o modelo dar nota maior a um positivo sorteado ao acaso do que a um
negativo sorteado ao acaso — e $0{,}85$ está longe de $0{,}5$, que seria o acaso
puro. O modelo sabe alguma coisa; a acurácia é que não sabe mostrar.

É esse o assunto da Aula 08: que pergunta cada métrica responde, o que muda quando se
mexe no corte de $0{,}5$, e por que em classe rara a curva de precisão–revocação diz
mais do que a ROC.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| sensibilidade à escala | §2 | MQO e árvores não mudam; Ridge e KNN mudam — padronize quem soma colunas |
| vazamento por seleção | §3 | $R^2 = +0{,}40$ a partir de **ruído puro**; feito certo, $-0{,}69$ |
| vazamento por escala | §4 | $0{,}8466$ contra $0{,}8466$: numericamente irrelevante |
| vazamento por agrupamento | §5 | `KFold` mente quando a mesma unidade está nos dois lados; use `GroupKFold` |
| hierarquia | §5 | seleção e agrupamento são graves; escala é leve — mas corrija todos |
| `Pipeline` | §6 | o scaler aprende a média **do treino**; o teste nunca entra num `fit` |
| `ColumnTransformer` | §7 | numéricas e categóricas em ramos separados; `handle_unknown="ignore"` |
| busca no pré-processamento | §8 | `prep__num__imp__strategy` é hiperparâmetro como qualquer outro |
| arquivo quebrado | §9 | `var_199;;;;;;;` viraria lixo silencioso sem o diagnóstico |

Se sobrar uma única frase desta aula, que seja a pergunta que separa o grave do
leve: **essa etapa olhou o $Y$?**

**Leitura recomendada.** [AME] §2.1.2 (o papel do pré-processamento na estimação).
[ISLP] os laboratórios dos Capítulos 5 e 6, onde o `Pipeline` aparece pela primeira
vez, e a discussão do Capítulo 6 sobre por que a seleção de variáveis precisa estar
dentro da validação cruzada — que é exatamente a medição da Seção 3. Vale também o
[ESL] §7.10.2, que é a origem desse exemplo.

**Para praticar.** `Lista teorica 06.pdf` (teórica, com gabarito) e
`Lista prática 06.ipynb` (prática, para completar as lacunas), nesta mesma
pasta.

**A seguir.** Fecha o Bloco I. A Aula 07 abre a classificação: a resposta deixa de
ser um número e passa a ser uma classe, o risco deixa de ser erro quadrático, e
quase tudo o que construímos até aqui precisa ser reescrito — com a vantagem de que
agora sabemos exatamente o que estamos reescrevendo.